# EDA - Exploratory Data Analysis
## Dataset: Mohler Short Answer Grading
**Goal:** Find non-obvious patterns that dictate modeling strategy.

## 1. Setup & Load Data

In [ ]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import shapiro, pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
from datasets import load_dataset
ds = load_dataset('nkazi/MohlerASAG')
records = []
for split in ds.keys():
    for row in ds[split]:
        if not row.get('id'): continue
        q_id = '.'.join(row['id'].split('.')[:2])
        records.append({
            'question_id': q_id,
            'answer': str(row['student_answer']).replace('<br>','').strip(),
            'score': float(row.get('score_avg', 0))
        })
df = pd.DataFrame(records)
df['word_count'] = df['answer'].str.split().str.len()
df['label'] = (df['score'] >= 3).astype(int)
print(f"Dataset: {len(df)} samples, {df.question_id.nunique()} questions")

## 2. Key Finding 1: TF-IDF Overlap Zone

In [ ]:
ref_answers = df.groupby('question_id')['answer'].first().to_dict()
def get_tfidf_sim(row):
    ref = ref_answers.get(row['question_id'], '')
    if not ref: return np.nan
    try:
        vec = TfidfVectorizer().fit_transform([ref, row['answer']])
        return float(cosine_similarity(vec[0], vec[1])[0,0])
    except: return np.nan
df['tfidf_sim'] = df.apply(get_tfidf_sim, axis=1)

thresholds = np.arange(0.05, 0.95, 0.01)
y_true = df['label'].values
f1_scores = [f1_score(y_true, (df['tfidf_sim'] >= t).astype(int), zero_division=0) for t in thresholds]
best_thr = thresholds[np.argmax(f1_scores)]
best_f1 = max(f1_scores)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
PALETTE = ["#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F"]

ax = axes[0]
for label, color, name in [(1, PALETTE[0], 'Pass'), (0, PALETTE[2], 'Fail')]:
    subset = df[df['label'] == label]['tfidf_sim'].dropna()
    ax.hist(subset, bins=25, alpha=0.6, color=color, label=name, density=True)
ax.axvspan(0.2, 0.5, alpha=0.15, color='gray', label='Overlap')
ax.set_xlabel('TF-IDF Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('KEY: Score bands overlap in TF-IDF space')
ax.legend()

ax = axes[1]
ax.plot(thresholds, f1_scores, color=PALETTE[0], lw=2)
ax.axvline(best_thr, color=PALETTE[2], linestyle='--', lw=2, label=f'Optimal: {best_thr:.2f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('F1 Score')
ax.set_title(f'TF-IDF F1 ceiling: {best_f1:.4f}')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\nKEY FINDING 1: Overlap zone (0.2-0.5) contains {len(df[(df["tfidf_sim"]>=0.2)&(df["tfidf_sim"]<=0.5)])} samples')

## 3. Key Finding 2: Length Confound

In [ ]:
r, p = pearsonr(df['word_count'], df['score'])
plt.figure(figsize=(6, 4))
plt.scatter(df['word_count'], df['score'], alpha=0.2, s=10, color=PALETTE[0])
plt.xlabel('Word Count'); plt.ylabel('Score')
plt.title(f'KEY FINDING 2: Length Confound (r={r:.3f})')
plt.tight_layout()
plt.show()
print(f'Length correlates with score: r={r:.3f}')

## 4. Key Finding 3: Class Imbalance & Question Difficulty

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df['label'].value_counts().sort_index()
ax = axes[0]
ax.bar(['Fail', 'Pass'], [counts[0], counts[1]], color=[PALETTE[2], PALETTE[0]], edgecolor='white')
ax.set_ylabel('Count')
ax.set_title(f'Class Distribution ({counts.max()/counts.min():.1f}:1 imbalance)')

q_stats = df.groupby('question_id')['score'].mean().sort_values()
ax = axes[1]
colors = [PALETTE[2] if s < q_stats.mean() else PALETTE[0] for s in q_stats.values]
ax.barh(range(len(q_stats)), q_stats.values, color=colors)
ax.set_yticks(range(len(q_stats)))
ax.set_yticklabels(q_stats.index)
ax.set_xlabel('Mean Score')
ax.set_title('Question Difficulty (Hard=Red)')

plt.tight_layout()
plt.show()

print(f'\nKEY FINDING 3: Class imbalance {counts.max()/counts.min():.1f}:1, Question difficulty varies {q_stats.min():.2f}-{q_stats.max():.2f}')

## 5. Normality Test

In [ ]:
print('Normality Test (Shapiro-Wilk):')
for col in ['tfidf_sim', 'word_count']:
    vals = df[col].dropna().values
    stat, p = shapiro(vals[:500])
    print(f'  {col}: p={p:.4f} -> {"Normal" if p>0.05 else "Non-normal"}')